In [ ]:
!pip install -q "transformers>=4.42" accelerate sentencepiece \
  sentence-transformers neo4j bitsandbytes


In [ ]:
import os
import time
import logging
from threading import Thread

import numpy as np
import torch
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TextIteratorStreamer,
)

from transformers import BitsAndBytesConfig  # ok if 4-bit not used

# ---------- Logging ----------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] [%(name)s] %(message)s",
)
log = logging.getLogger("HYBRID-RAG-COLAB")

STEP_TIMES = {}

def record_step(name: str, start: float):
    STEP_TIMES[name] = time.perf_counter() - start

def print_summary():
    log.info("\n========== PIPELINE TIMING SUMMARY ==========")
    total = 0.0
    for step, sec in STEP_TIMES.items():
        log.info(f"{step:30s}: {sec:.3f}s")
        total += sec
    log.info("----------------------------------------------")
    log.info(f"TOTAL PIPELINE TIME            : {total:.3f}s")
    log.info("==============================================\n")


In [ ]:
# ------- Speed/quality knobs -------
FAST_MODE = True          # True = TinyLlama (fast), False = Phi-3 (better quality)
QUANTIZE_4BIT = True      # only used when FAST_MODE=False and GPU available
MAX_NEW_TOKENS_FAST = 96
MAX_NEW_TOKENS_QUALITY = 256
MAX_DOC_CHARS = 800       # truncate each chunk
MAX_CONTEXT_TOKENS = 4000 # rough context window

# ------- Environment check -------
step = time.perf_counter()
device = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device selected: {device}")
if device == "cuda":
    log.info(f"GPU:  {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
else:
    log.warning("Running on CPU — expect slow LLM inference.")

# ------- Neo4j connection (Aura) -------
# NEO4J_URI      = os.getenv("NEO4J_URI",      "neo4j+s://YOUR_AURA_DB.databases.neo4j.io")
# NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
# NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
# --- Neo4j Aura Credentials ---
import os
NEO4J_URI      = os.getenv("NEO4J_URI",      "neo4j+s://5db7d1f6.databases.neo4j.io")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "hCBE_-lRkn_A_02LIVIpnppdQ9V6n0BXVW6rYuC7srA")
neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

# ------- Embedding model -------
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL_NAME)
EMB_DIM = emb_model.get_sentence_embedding_dimension()
log.info(f"Embedding model: {EMB_MODEL_NAME} (dim={EMB_DIM})")

record_step("Env + Neo4j + Embeddings", step)

def embed(texts):
    t0 = time.perf_counter()
    vecs = emb_model.encode(texts, normalize_embeddings=False)
    t1 = time.perf_counter()
    log.info(f"Embedded {len(texts)} text(s) in {t1 - t0:.3f}s")
    return vecs.tolist()

def cosine_similarity(u, v):
    u = np.array(u, dtype=np.float32)
    v = np.array(v, dtype=np.float32)
    num = float(np.dot(u, v))
    den = float(np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)
    return num / den


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [ ]:
step = time.perf_counter()

if FAST_MODE:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    MAX_NEW_TOKENS = MAX_NEW_TOKENS_FAST
    log.info("LLM mode: FAST (TinyLlama)")
else:
    MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
    MAX_NEW_TOKENS = MAX_NEW_TOKENS_QUALITY
    log.info("LLM mode: QUALITY (Phi-3)")

log.info(f"Loading LLM: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if (not FAST_MODE) and QUANTIZE_4BIT and device == "cuda":
    log.info("Using 4-bit quantization for QUALITY mode.")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    model.to(device)

model.eval()
num_params = sum(p.numel() for p in model.parameters())
log.info(f"LLM params: {num_params/1e9:.3f}B")

record_step("LLM Loading", step)


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
HYBRID_QUERY = """
CALL {
    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    RETURN n.node AS node, (n.score / max) AS score

    UNION

    CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit: $k})
    YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    RETURN n.node AS node, (n.score / max) AS score
}
WITH node, max(score) AS score
ORDER BY score DESC
LIMIT $k
RETURN node, score
"""

def hybrid_search_neo4j(question: str, k: int = 3):
    step = time.perf_counter()
    log.info(f"STEP: Hybrid Search — question={question!r}, k={k}")

    q_emb = embed([question])[0]

    records, summary, keys = neo4j_driver.execute_query(
        HYBRID_QUERY,
        question_embedding=q_emb,
        question=question,
        k=k,
    )

    hybrid_records = []
    for i, rec in enumerate(records):
        node = rec["node"]
        text = node["text"]
        hybrid_score = rec["score"]
        node_emb = node.get("embedding")

        cos_sim = cosine_similarity(q_emb, node_emb) if node_emb is not None else None
        truncated_text = text[:MAX_DOC_CHARS]

        hybrid_records.append(
            {
                "text": truncated_text,
                "full_text_len": len(text),
                "score": hybrid_score,
                "cosine_similarity": cos_sim,
                "node": node,
            }
        )

        log.info(
            f"Hit[{i}] hybrid={hybrid_score:.4f} "
            f"cos={cos_sim:.4f} full_len={len(text)} trunc_len={len(truncated_text)}"
            if cos_sim is not None
            else f"Hit[{i}] hybrid={hybrid_score:.4f} full_len={len(text)} trunc_len={len(truncated_text)}"
        )

    record_step("Hybrid Search (Neo4j)", step)
    return hybrid_records


In [ ]:
def build_rag_prompt(similar_records, question: str):
    step = time.perf_counter()
    log.info("STEP: Build Prompt")

    docs = [r["text"] for r in similar_records]
    for i, d in enumerate(docs[:3]):
        log.info(f"Doc[{i}] length (truncated): {len(d)} chars")

    docs_block = "\n\n---\n\n".join(docs)

    system_message = (
        "You are a helpful assistant. Use ONLY the provided documents. "
        "If the answer is not in them, say you don't know."
    )

    user_message = f"""
Use the following documents to answer the question that will follow:

{docs_block}

---

The question to answer using ONLY the above documents is:
{question}
""".strip()

    log.info(f"User message length: {len(user_message)} chars")
    record_step("Build Prompt", step)
    return system_message, user_message


def tokenize_and_check(prompt: str):
    step = time.perf_counter()
    log.info("STEP: Tokenization + Context Check")

    inputs = tokenizer(prompt, return_tensors="pt")
    seq_len = inputs["input_ids"].shape[1]
    pct = seq_len / MAX_CONTEXT_TOKENS * 100

    log.info(f"Prompt tokens: {seq_len}/{MAX_CONTEXT_TOKENS} ({pct:.1f}%)")
    if seq_len > MAX_CONTEXT_TOKENS:
        log.warning("⚠️ Prompt exceeds context window — will be truncated.")
    elif pct > 80:
        log.warning("⚠️ Prompt uses >80% of context — little room for generation.")

    record_step("Tokenization", step)
    return inputs.to(device), seq_len


def local_stream(system_message, user_message, max_new_tokens=None, max_time_sec: float = 30.0):
    """
    Stream answer safely:
      - max_new_tokens caps length
      - max_time_sec stops generation if it runs too long
    """
    step = time.perf_counter()
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS

    log.info(
        f"STEP: Inference — max_new_tokens={max_new_tokens}, max_time={max_time_sec}s"
    )

    prompt = f"<|system|>\n{system_message}\n<|user|>\n{user_message}\n<|assistant|>\n"
    inputs, _ = tokenize_and_check(prompt)

    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id,
        max_time=max_time_sec,   # ⏱ hard cap on generation time
    )

    def _gen():
        try:
            model.generate(**gen_kwargs)
        except Exception as e:
            log.exception(f"Generation error: {e}")

    Thread(target=_gen, daemon=True).start()

    t0 = time.perf_counter()
    token_count = 0
    char_count = 0

    for piece in streamer:
        print(piece, end="", flush=True)
        token_count += 1
        char_count += len(piece)

    t1 = time.perf_counter()
    elapsed = t1 - t0
    tps = token_count / elapsed if elapsed else 0.0

    log.info(f"\nGenerated tokens: {token_count}, chars: {char_count}")
    log.info(f"Inference time: {elapsed:.2f}s → {tps:.2f} tokens/s")

    record_step("Inference", step)


In [ ]:
def answer_with_hybrid_rag(question: str, k: int = 3):
    """
    End-to-end:
      1) hybrid retrieval from Neo4j
      2) RAG prompt build
      3) safe streaming answer
    """
    log.info("STEP: Pipeline Start — Hybrid RAG Query")
    hits = hybrid_search_neo4j(question, k=k)

    if not hits:
        log.warning("No hybrid records from Neo4j.")
        return

    system_msg, user_msg = build_rag_prompt(hits, question)

    print("\n📌 QUESTION:", question)
    print("\n🧠 ANSWER (Hybrid RAG):\n")
    local_stream(system_msg, user_msg)


# ---- Test run (small, safe) ----
test_question = "Explain the main idea of the document."
answer_with_hybrid_rag(test_question, k=3)
print_summary()
